In [46]:
import io
import requests
import PyPDF2
import pdfplumber
import re
import pytesseract
from PIL import Image
pytesseract.pytesseract.tesseract_cmd = r"C:\\Program Files\\Tesseract-OCR\\tesseract.exe"

In [47]:
def get_file_id_from_url(url):
    
    patterns = [
        r'/file/d/([a-zA-Z0-9-_]+)',
        r'id=([a-zA-Z0-9-_]+)',
        r'/d/([a-zA-Z0-9-_]+)',
    ]
    
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    
    return None

def get_direct_download_url(file_id):
    return f"https://drive.google.com/uc?export=download&id={file_id}"

In [48]:
def download_pdf_from_gdrive(url):
    file_id = get_file_id_from_url(url)
    
    if not file_id:
        print("Không thể trích xuất file ID từ URL")
        return None
    
    print(f"File ID: {file_id}")
    
    download_url = get_direct_download_url(file_id)
    
    session = requests.Session()
    response = session.get(download_url, stream=True)
    
    if 'download_warning' in response.text:
        for key, value in response.cookies.items():
            if key.startswith('download_warning'):
                params = {'id': file_id, 'confirm': value}
                response = session.get(download_url, params=params, stream=True)
                break
    
    if response.status_code == 200:
        print("Tải file thành công!")
        return io.BytesIO(response.content)
    
    print(f"Lỗi khi tải file. Status code: {response.status_code}")
    return None

In [49]:
def extract_text_from_pdf(pdf_buffer):
    try:
        text = ""
        with pdfplumber.open(pdf_buffer) as pdf:
            num_pages = len(pdf.pages)
            print(f"\nSố trang trong PDF: {num_pages}\n")
            print("=" * 70)
            for i, page in enumerate(pdf.pages):
                base_text = page.extract_text() or ""
                base_text = base_text.strip()
                image = page.to_image(resolution=300).original
                ocr_text = pytesseract.image_to_string(image, lang="vie") or ""
                ocr_text = ocr_text.strip()
                if not base_text and not ocr_text:
                    print(f"❌ Trang {i + 1} không trích xuất được text")
                    continue
                parts = []
                if base_text:
                    parts.append(base_text)
                if ocr_text and ocr_text not in base_text:
                    parts.append(ocr_text)
                page_text = "\n".join(parts)
                text += f"\n{'=' * 70}\n"
                text += f"TRANG {i + 1}/{num_pages}\n"
                text += f"{'=' * 70}\n"
                text += page_text
                text += "\n"
        return text
    except Exception as e:
        print(f"Lỗi khi đọc PDF: {str(e)}")
        return None

In [50]:
gdrive_urls = [
    "https://drive.google.com/file/d/1EUvhjkTFRLpiwDQ1y0DJgdevyIBwaeuV/view",
    "https://drive.google.com/file/d/1XHP5nQYg75v7Zw7P_SdwfNYPJg13xxPQ/view"
 ]

extracted_texts = {}

for idx, url in enumerate(gdrive_urls, start=1):
    print("\n" + "#" * 80)
    print(f"FILE {idx}: {url}")
    print("#" * 80)
    
    pdf_buffer = download_pdf_from_gdrive(url)
    
    if not pdf_buffer:
        print("Không thể tải file.")
        continue
    
    text = extract_text_from_pdf(pdf_buffer)
    
    if not text:
        print("Không trích xuất được nội dung.")
        continue
    
    key = get_file_id_from_url(url) or f"file_{idx}"
    extracted_texts[key] = text
    
    print("\n" + "=" * 70)
    print(f"NỘI DUNG FILE: {key}")
    print("=" * 70)
    print(text)


################################################################################
FILE 1: https://drive.google.com/file/d/1EUvhjkTFRLpiwDQ1y0DJgdevyIBwaeuV/view
################################################################################
File ID: 1EUvhjkTFRLpiwDQ1y0DJgdevyIBwaeuV
Tải file thành công!

Số trang trong PDF: 35


NỘI DUNG FILE: 1EUvhjkTFRLpiwDQ1y0DJgdevyIBwaeuV

TRANG 1/35
CHAPTER 1
Extracting the Data
In this chapter, we are going to cover various sources of text data and ways
to extract it, which can act as information or insights for businesses.
Recipe 1. Text data collection using APIs
Recipe 2. Reading PDF file in Python
Recipe 3. Reading word document
Recipe 4. Reading JSON object
Recipe 5. Reading HTML page and HTML parsing
Recipe 6. Regular expressions
Recipe 7. String handling
Recipe 8. Web scraping
I ntroduction
Before getting into details of the book, let’s see the different possible data
sources available in general. We need to identify potential data sourc

In [51]:
if 'extracted_texts' in globals() and extracted_texts:
    for key, text in extracted_texts.items():
        output_file = f"{key}_extracted.txt"
        
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(text)
        
        print(f"✓ Đã lưu văn bản file {key} vào: {output_file}")
        print(f"  Tổng số ký tự: {len(text):,}")
else:
    print("Không có dữ liệu để lưu.")

✓ Đã lưu văn bản file 1EUvhjkTFRLpiwDQ1y0DJgdevyIBwaeuV vào: 1EUvhjkTFRLpiwDQ1y0DJgdevyIBwaeuV_extracted.txt
  Tổng số ký tự: 65,179
✓ Đã lưu văn bản file 1XHP5nQYg75v7Zw7P_SdwfNYPJg13xxPQ vào: 1XHP5nQYg75v7Zw7P_SdwfNYPJg13xxPQ_extracted.txt
  Tổng số ký tự: 96,107


In [52]:
if 'extracted_texts' in globals() and extracted_texts:
    print("\n" + "=" * 50)
    print("THỐNG KÊ VĂN BẢN THEO FILE")
    print("=" * 50)
    
    for key, text in extracted_texts.items():
        words = text.split()
        lines = text.split('\n')
        
        print("\n" + "-" * 40)
        print(f"FILE: {key}")
        print("-" * 40)
        print(f"Tổng số ký tự: {len(text):,}")
        print(f"Tổng số từ: {len(words):,}")
        print(f"Tổng số dòng: {len(lines):,}")
        print(f"Trung bình ký tự/dòng: {len(text)/max(len(lines),1):.1f}")
    
    print("\n" + "=" * 50)
else:
    print("Không có dữ liệu để thống kê.")


THỐNG KÊ VĂN BẢN THEO FILE

----------------------------------------
FILE: 1EUvhjkTFRLpiwDQ1y0DJgdevyIBwaeuV
----------------------------------------
Tổng số ký tự: 65,179
Tổng số từ: 8,902
Tổng số dòng: 2,370
Trung bình ký tự/dòng: 27.5

----------------------------------------
FILE: 1XHP5nQYg75v7Zw7P_SdwfNYPJg13xxPQ
----------------------------------------
Tổng số ký tự: 96,107
Tổng số từ: 20,009
Tổng số dòng: 2,964
Trung bình ký tự/dòng: 32.4

